# Lasso â€” L1-Regularized Regression (Pipeline B)

**Model**: Lasso via `sklearn.linear_model.Lasso`
**Target**: `gdpc1`
**Variables**: Cat3 (53 vars from L95\u222aE95\u222aR95\u222aS100 + COVID = 56 total)
**Train**: 1959-Q1 to 2007-Q4 / **Val**: 2008-Q1 to 2016-Q4
**Test**: 2017-Q1 to 2025-Q4 (36 quarters, m1/m2/m3)
**Scaling**: YES (StandardScaler, fit on train fold only)
**HP tuning**: YES â€” `LassoCV(cv=TimeSeriesSplit(5))` on train+val, freeze alpha
**n_lags**: 4
**COVID**: Passed through unscaled via `split_for_scaler()`


In [1]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LassoCV, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from scipy.stats import norm

plt.rcParams["figure.figsize"] = [15, 10]

sys.path.insert(0, os.path.join(os.path.pardir, "data"))
from helpers import (gen_lagged_data, gen_vintage_data, make_supervised_vintage_frame, flatten_data, mean_fill_dataset,
                     get_features, split_for_scaler, load_data,
                     PREDICTIONS_DIR, TARGET, COVID)

SEED = 42
np.random.seed(SEED)

TRAIN_START = "1959-01-01"
TRAIN_END   = "2007-12-31"
VAL_END     = "2016-12-31"
TEST_START  = "2017-01-01"
TEST_END    = "2025-12-31"
N_LAGS = 4
VINTAGES = {"m1": -2, "m2": -1, "m3": 0, "post1": 1}

print("Lasso \u2014 Cat3 variables, n_lags={}, scaling=YES, HP tune=YES".format(N_LAGS))


Lasso — Cat3 variables, n_lags=4, scaling=YES, HP tune=YES


In [2]:
monthly, _, metadata = load_data()
features = get_features("cat3", with_covid=True)
scaled_cols, unscaled_cols = split_for_scaler(features)
print("Features: {} total ({} scaled, {} COVID passed through)".format(
    len(features), len(scaled_cols), len(unscaled_cols)))


Features: 56 total (53 scaled, 3 COVID passed through)

In [3]:
# Phase A: HP tuning on train+val (1959-2016) via TimeSeriesSplit
print("Phase A: HP tuning on train+val (1959-2016)...")

tune_data = monthly[(monthly["date"] >= TRAIN_START) & (monthly["date"] <= VAL_END)].reset_index(drop=True)
tune_filled = mean_fill_dataset(tune_data, tune_data)
tune_flat = flatten_data(tune_filled, TARGET, N_LAGS)
tune_flat = tune_flat.loc[tune_flat.date.dt.month.isin([3, 6, 9, 12]), :]
tune_flat = tune_flat.dropna(axis=0, how="any").reset_index(drop=True)

feature_cols = [c for c in tune_flat.columns if c != "date" and c != TARGET and any(c == f or c.startswith(f + "_") for f in features)]
X_tune = tune_flat[feature_cols].values
y_tune = tune_flat[TARGET].values

# Keep scaling inside each TimeSeriesSplit fold to avoid validation-fold leakage.
alpha_path = np.logspace(-6, 0, 100)
tscv = TimeSeriesSplit(n_splits=5)
lasso_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LassoCV(alphas=alpha_path, cv=tscv, random_state=SEED,
                       max_iter=20000, n_jobs=-1)),
])
lasso_pipe.fit(X_tune, y_tune)
lasso_cv = lasso_pipe.named_steps["model"]
best_alpha = lasso_cv.alpha_
print("Best alpha: {:.4g}".format(best_alpha))


Phase A: HP tuning on train+val (1959-2016)...


Best alpha: 2.477e-05


In [4]:
# Phase B: Rolling test loop with frozen alpha
test_quarters = monthly[(monthly["date"] >= TEST_START) &
                         (monthly["date"] <= TEST_END) &
                         monthly["date"].dt.month.isin([3, 6, 9, 12])]["date"].tolist()

predictions = {v: [] for v in VINTAGES}
actuals_list = []
failed = 0

for i, q_date in enumerate(test_quarters):
    pd_q = pd.Timestamp(q_date)
    actual = monthly[monthly["date"] == pd_q][TARGET].values[0]
    actuals_list.append(actual)

    for vintage_name, offset in VINTAGES.items():
        forecast_date = pd_q + pd.DateOffset(months=offset)
        train_flat, test_flat, _ = make_supervised_vintage_frame(
            metadata, monthly, TARGET, features, TRAIN_START, pd_q, forecast_date, N_LAGS
        )

        if len(train_flat) < 10:
            predictions[vintage_name].append(np.nan)
            continue

        # Separate scaled/unscaled
        sc_cols = [c for c in feature_cols if c in scaled_cols or any(c.startswith(s) for s in scaled_cols)]
        us_cols = [c for c in feature_cols if c in unscaled_cols or any(c.startswith(u) for u in unscaled_cols)]

        try:
            scaler = StandardScaler()
            X_train_sc = scaler.fit_transform(train_flat[sc_cols].values)
            X_train = np.hstack([X_train_sc, train_flat[us_cols].values]) if us_cols else X_train_sc
            y_train = train_flat[TARGET].values

            model = Lasso(alpha=best_alpha, max_iter=20000, random_state=SEED)
            model.fit(X_train, y_train)

            X_test_sc = scaler.transform(test_flat[sc_cols].values)
            X_test = np.hstack([X_test_sc, test_flat[us_cols].values]) if us_cols else X_test_sc
            pred = model.predict(X_test)[0]
        except Exception:
            pred = np.nan
            failed += 1

        predictions[vintage_name].append(pred)

    if (i + 1) % 8 == 0:
        print("  {} / {} quarters".format(i + 1, len(test_quarters)))

print("Done. {} quarters, {} failures.".format(len(test_quarters), failed))


  8 / 36 quarters


  16 / 36 quarters


  24 / 36 quarters


  32 / 36 quarters


Done. 36 quarters, 0 failures.


In [5]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
for vn in VINTAGES:
    df = pd.DataFrame({
        "date": test_quarters, "actual": actuals_list,
        "prediction": predictions[vn],
    })
    path = os.path.join(PREDICTIONS_DIR, "lasso_{}.csv".format(vn))
    df.to_csv(path, index=False)
    print("Saved {}".format(path))


Saved C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\predictions\lasso_m1.csv
Saved C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\predictions\lasso_m2.csv
Saved C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\predictions\lasso_m3.csv
Saved C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\predictions\lasso_post1.csv


In [6]:
def rmse(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.sqrt(np.mean((a[m]-p[m])**2)) if m.sum()>0 else np.nan

def mae(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.mean(np.abs(a[m]-p[m])) if m.sum()>0 else np.nan

panels = {
    "Pre-COVID  (2017-2019)": ("2017-01-01", "2019-12-31"),
    "COVID      (2020-2021)": ("2020-04-01", "2021-12-31"),
    "Post-COVID (2022-2025)": ("2022-01-01", "2025-12-31"),
    "Full test  (2017-2025)": ("2017-01-01", "2025-12-31"),
}
a = np.array(actuals_list)
d = pd.to_datetime(test_quarters)
print("Lasso alpha = {:.4g}".format(best_alpha))
for pn, (ps, pe) in panels.items():
    m = (d >= ps) & (d <= pe)
    print("--- {} ---".format(pn))
    for vn in VINTAGES:
        pp = np.array(predictions[vn])
        print("  {}  RMSFE={:.6f}  MAE={:.6f}  N={}".format(vn, rmse(a[m],pp[m]), mae(a[m],pp[m]), m.sum()))


Lasso alpha = 2.477e-05
--- Pre-COVID  (2017-2019) ---
  m1  RMSFE=0.002713  MAE=0.002137  N=12
  m2  RMSFE=0.002732  MAE=0.002063  N=12
  m3  RMSFE=0.002703  MAE=0.002001  N=12
  post1  RMSFE=0.002702  MAE=0.001991  N=12
--- COVID      (2020-2021) ---
  m1  RMSFE=0.043706  MAE=0.027414  N=7
  m2  RMSFE=0.041985  MAE=0.026353  N=7
  m3  RMSFE=0.041379  MAE=0.025921  N=7
  post1  RMSFE=0.041081  MAE=0.025855  N=7
--- Post-COVID (2022-2025) ---
  m1  RMSFE=0.004486  MAE=0.003363  N=16
  m2  RMSFE=0.004492  MAE=0.003420  N=16
  m3  RMSFE=0.004523  MAE=0.003451  N=16
  post1  RMSFE=0.004443  MAE=0.003438  N=16
--- Full test  (2017-2025) ---
  m1  RMSFE=0.019869  MAE=0.008114  N=36
  m2  RMSFE=0.019133  MAE=0.007906  N=36
  m3  RMSFE=0.018868  MAE=0.007807  N=36
  post1  RMSFE=0.018759  MAE=0.007809  N=36
